# 17-ncc-dedup-check, near-duplicate cross-check with normalised cross-correlation

Settings: **Accelerator None (CPU)**, **Internet On**, secret `kaggle_dpfl-braintumor` attached. Inputs: the dataset `brain-mri-curated-v1` (Part A), and for Part B the same raw inputs as notebook 02: the Figshare zips (output of 01-data-acquisition) and the Kaggle datasets sartajbhuvaji/brain-tumor-classification-mri and ahmedhamada0/brain-tumor-detection.

Runs code tag `v0.7-ncc`. Uses no GPU quota. Results go to `results/ncc/`.

Generated by `scripts/make_launcher.py --kind ncc`, edit that script rather than this notebook.

In [ ]:
import os, sys, glob, subprocess, datetime, shutil
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
CODE_REF = "v0.7-ncc"
token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
os.environ["GIT_PUSH_URL"] = f"https://{token}@github.com/{USER}/{REPO}.git"
CLEAN = f"https://github.com/{USER}/{REPO}.git"
CODE, RES = "/tmp/code", "/tmp/results"

def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out[-3000:])
    return out

for d in (CODE, RES):
    shutil.rmtree(d, ignore_errors=True)
    run(["git", "clone", "-q", os.environ["GIT_PUSH_URL"], d])
    run(["git", "remote", "set-url", "origin", CLEAN], cwd=d)
run(["git", "checkout", "-q", CODE_REF], cwd=CODE)
run(["git", "config", "user.name", USER], cwd=RES)
run(["git", "config", "user.email", EMAIL], cwd=RES)
sys.path.insert(0, CODE)
from scripts.run_grid import push_run

hits = glob.glob("/kaggle/input/**/brain_mri_curated_v1.npz", recursive=True)
if not hits:
    raise SystemExit("Attach the dataset brain-mri-curated-v1 through Add Input, then rerun this cell.")
DATA = os.path.dirname(hits[0])
mats = glob.glob("/kaggle/input/**/brainTumorDataPublic_*.zip", recursive=True) + \
       glob.glob("/kaggle/input/**/*.mat", recursive=True)
dirs = [r for r, _, _ in os.walk("/kaggle/input")
        if os.path.basename(r) in ("brain-tumor-classification-mri", "brain-tumor-detection")]
print("code", CODE_REF, run(["git", "rev-parse", "--short", "HEAD"], cwd=CODE).strip(), "| curated", DATA)
print("raw Figshare files found:", len(mats), "| raw folders found:", sorted({os.path.basename(d) for d in dirs}))
print("Part B needs Figshare zips or .mat files and both folders; Part A needs only the curated dataset.")

In [ ]:
# Part A: all pairs of the 3845 curated images, NCC on 64 x 64, plain and flipped. A few minutes on CPU.
print(run([sys.executable, "scripts/ncc_check.py", "curated", "--data-dir", DATA,
           "--out", f"{RES}/results/ncc"], cwd=CODE)[-6000:])
print("push", push_run(RES, "results/ncc", f"NCC check, curated set ({CODE_REF})"))
from IPython.display import Image, display
p = f"{RES}/results/ncc/top_cross_split_pairs.png"
if os.path.exists(p):
    display(Image(p))

In [ ]:
# Part B: every removed duplicate against the image it was merged into, rebuilt from the raw sources.
out = run([sys.executable, "scripts/ncc_check.py", "removed", "--raw-dirs", "/kaggle/input",
           "--out", f"{RES}/results/ncc"], cwd=CODE, check=False)
print(out[-6000:])
print("push", push_run(RES, "results/ncc", f"NCC check, removed duplicates ({CODE_REF})"))